# DDT vs Boosting: California Housing Benchmark Profiles

Compares **DDT (Distributional Decision Tree)** against LightGBM, XGBoost, and CatBoost
on the [California Housing dataset](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) (~20,640 rows, 8 features).

**Dataset notes:**
- Cross-sectional census data (no temporal ordering) — a random 80/20 split is appropriate.
- Target is median house value in $100k units, **hard-capped at $500k** in the original census.
  This creates a known spike at the 95th-ish percentile ceiling — a good stress test for tail calibration.

**Measurement principles:**
- CVaR uses a fixed empirical threshold (true P95 of the test set), not the per-sample model P95.
- Model size via `pickle.dumps` (note: LGBM pickle includes Python wrapper overhead).
- Feature scaling: tree models are invariant to monotone transforms; `"none"` is the default.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, TargetEncoder
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from ddt import DDTRegressor
import time
import pickle
from IPython.display import display, HTML

# --- Metrics ---
def calc_cce(y_true, q_lower, q_upper, target_coverage):
    empirical_coverage = np.mean((y_true >= q_lower) & (y_true <= q_upper))
    return abs(target_coverage - empirical_coverage)

def calc_apiw(q_lower, q_upper):
    return np.mean(q_upper - q_lower)

def calc_cvar(y_true, q_pred):
    """CVaR computed with the 95% of the predicted."""
    breaches = y_true[y_true > q_pred]
    return float(np.mean(breaches)) if len(breaches) > 0 else 0.0

def calc_hei(infer_time_sec, n_samples, model_obj):
    latency_us = (infer_time_sec / n_samples) * 1e6
    size_kb = len(pickle.dumps(model_obj)) / 1024
    return latency_us * size_kb


In [2]:
BENCHMARK_PROFILES = {
    "1_vs_1_theory": {
        "name": "1-vs-1 Representational Capacity (The Theoretical Duel)",
        "description": (
            "Intrinsic capacity of a single tree without ensembling. "
            "All models: 1 tree per quantile. Exposes the Flaw of Averages head-on."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 5, "min_samples_leaf": 50,
            "n_target_bins": 64, "n_feature_bins": 32,
        },
        # DDT-Cal: DDT-Hybrid + marginal conformal calibration.
        # Adjust calibration_fraction / calibration_mode to experiment.
        "ddt_cal_params": {
            "calibration_fraction": 0.15,   # fraction held out for calibration
            "calibration_mode": "hybrid",  # 'marginal' | 'hybrid' (leaf-level)
            "core_fraction": 0.5,            # hybrid-bin core fraction
        },
        "competitors_params": {
            "n_estimators": 1, "max_depth": 5,
        }
    },

    "edge_microkernel": {
        "name": "Edge AI & Micro-Kernel (The Silicon Test)",
        "description": (
            "Constrained hardware (IoT/HFT). All models: 1 tree per quantile (true 1-vs-1). "
            "DDT produces all quantiles in one pass; GBMs use separate per-quantile trees."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 5, "min_samples_leaf": 100,
            "n_target_bins": 64, "n_feature_bins": 32,
        },
        # DDT-Cal: DDT-Hybrid + marginal conformal calibration.
        # Adjust calibration_fraction / calibration_mode to experiment.
        "ddt_cal_params": {
            "calibration_fraction": 0.15,   # fraction held out for calibration
            "calibration_mode": "hybrid",  # 'marginal' | 'hybrid' (leaf-level)
            "core_fraction": 0.5,            # hybrid-bin core fraction
        },
        "competitors_params": {
            # Fixed: was 3 (unfair vs DDT single tree). Now 1 per quantile.
            "n_estimators": 1, "max_depth": 5,
        }
    },

    "production_ensemble": {
        "name": "Production Battle (Single-Tree DDT vs. 100-Tree Boosting)",
        "description": (
            "Single DDT vs 100-tree GBM ensembles. "
            "GBMs train 3x100-tree models (one per quantile): 300 trees total. "
            "Model size reflects full multi-quantile deployment cost."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 8, "min_samples_leaf": 200,
            "n_target_bins": 64, "n_feature_bins": 64,
        },
        # DDT-Cal: DDT-Hybrid + marginal conformal calibration.
        # Adjust calibration_fraction / calibration_mode to experiment.
        "ddt_cal_params": {
            "calibration_fraction": 0.15,   # fraction held out for calibration
            "calibration_mode": "hybrid",  # 'marginal' | 'hybrid' (leaf-level)
            "core_fraction": 0.5,            # hybrid-bin core fraction
        },
        "competitors_params": {
            "n_estimators": 100, "max_depth": 8,
        }
    },

    "ceiling_tail_stress": {
        "name": "Ceiling Tail Stress Test (DDT Core Strength)",
        "description": (
            "California Housing target is hard-capped at $500k in the original census, "
            "creating a spike at the upper tail (~5% of data). "
            "This tests whether models can correctly identify that the true P95 is near "
            "the ceiling and avoid overconfident narrow intervals. "
            "DDT captures the ceiling spike as a discrete mass in its ECDF; "
            "GBMs may regress through it. "
            "n_target_bins=80 gives full resolution across the capped distribution. "
            "GBMs receive 50 trees for competitive footing."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 7, "min_samples_leaf": 100,
            "n_target_bins": 80, "n_feature_bins": 64,
        },
        # DDT-Cal: DDT-Hybrid + marginal conformal calibration.
        # Adjust calibration_fraction / calibration_mode to experiment.
        "ddt_cal_params": {
            "calibration_fraction": 0.15,   # fraction held out for calibration
            "calibration_mode": "hybrid",  # 'marginal' | 'hybrid' (leaf-level)
            "core_fraction": 0.5,            # hybrid-bin core fraction
        },
        "competitors_params": {
            "n_estimators": 50, "max_depth": 7,
        }
    },
}


In [3]:
# --- DATA LOADING & PREPROCESSING ---
# California Housing is cross-sectional (no temporal ordering).
# A random 80/20 split is appropriate here.

from sklearn.preprocessing import MinMaxScaler
cal = fetch_california_housing(as_frame=True)
X = cal.data.copy()
y = cal.target.values.astype(float)

valid = ~np.isnan(y)
X = X[valid].reset_index(drop=True)
y = y[valid]

print(f"Dataset: {len(X):,} rows, {X.shape[1]} features")
print(f"Target range: [{y.min():.3f}, {y.max():.3f}] ($100k units)")
print(f"Rows at $500k ceiling: {(y >= 4.999).sum():,} ({(y >= 4.999).mean()*100:.1f}%)")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fixed CVaR threshold: true P95 of the test set (computed before any model runs)
EMPIRICAL_P95_TEST = float(np.percentile(y_test, 95))
print(f"Fixed CVaR threshold (true test P95): ${EMPIRICAL_P95_TEST*100:.0f}k")

categorical_cols = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols   = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()

def preprocess_pipeline(X_train_df, X_test_df, y_train_arr,
                         target_strategy="winsorized", feature_scaling="none"):
    X_tr = X_train_df.copy()
    X_te = X_test_df.copy()
    y_tr = y_train_arr.copy()

    # California Housing has no categoricals, but keep the branch for generality
    if len(categorical_cols) > 0:
        enc = TargetEncoder(smooth="auto", cv=5, random_state=42, target_type="continuous")
        X_tr_cat = enc.fit_transform(X_tr[categorical_cols], y_train_arr)
        X_te_cat = enc.transform(X_te[categorical_cols])
        X_tr_num = X_tr[numerical_cols].fillna(0).values
        X_te_num = X_te[numerical_cols].fillna(0).values
        X_train_out = np.hstack([X_tr_num, X_tr_cat])
        X_test_out  = np.hstack([X_te_num, X_te_cat])
    else:
        X_train_out = X_tr[numerical_cols].fillna(0).values
        X_test_out  = X_te[numerical_cols].fillna(0).values

    if feature_scaling == "standard":
        sc = StandardScaler()
        X_train_out = sc.fit_transform(X_train_out)
        X_test_out  = sc.transform(X_test_out)
    elif feature_scaling == "minmax":
        sc = MinMaxScaler()
        X_train_out = sc.fit_transform(X_train_out)
        X_test_out  = sc.transform(X_test_out)
    # "none": tree models are invariant to monotone feature transforms

    y_scaler = None
    if target_strategy == "winsorized":
        lo, hi = np.percentile(y_tr, 1), np.percentile(y_tr, 99)
        y_tr = np.clip(y_tr, lo, hi)
    if target_strategy in ["standard", "winsorized"]:
        y_scaler = StandardScaler()
        y_tr = y_scaler.fit_transform(y_tr.reshape(-1, 1)).flatten()

    return X_train_out, X_test_out, y_tr, y_scaler


Dataset: 20,640 rows, 8 features
Target range: [0.150, 5.000] ($100k units)
Rows at $500k ceiling: 992 (4.8%)
Fixed CVaR threshold (true test P95): $473k


In [4]:
def run_benchmark(profile_name, profile, X_train_scaled, X_test_scaled, y_train, y_test, preproc_name, y_scaler=None):
    sep = "=" * 80
    print(f"\n{sep}\nProfile: {profile['name']} | Preprocessing: {preproc_name}\n{sep}")
    n_test = len(y_test)
    ddt_p  = profile["ddt_params"].copy()
    comp_p = profile["competitors_params"].copy()

    # --- DDT ---
    ddt = DDTRegressor(**ddt_p)
    t0 = time.time(); ddt.fit(X_train_scaled, y_train); ddt_fit_t = time.time() - t0
    t0 = time.time()
    ddt_preds = ddt.predict_quantiles(X_test_scaled, profile["target_quantiles"])
    ddt_inf_t = time.time() - t0
    ddt_q05 = ddt_preds[profile["target_quantiles"][0]]
    ddt_q95 = ddt_preds[profile["target_quantiles"][2]]

    # --- DDT-Log ---
    ddt_log_p = ddt_p.copy()
    ddt_log_p["bin_strategy"] = "log_width"
    ddt_log_p["winsorize_tails"] = 0.001
    
    offset = abs(np.min(y_train)) + 1.0 if np.min(y_train) <= 0 else 0.0
    y_train_log = y_train + offset
    
    ddt_log = DDTRegressor(**ddt_log_p)
    t0 = time.time(); ddt_log.fit(X_train_scaled, y_train_log); ddt_log_fit_t = time.time() - t0
    t0 = time.time()
    ddt_log_preds = ddt_log.predict_quantiles(X_test_scaled, profile["target_quantiles"])
    ddt_log_inf_t = time.time() - t0
    ddt_log_q05 = ddt_log_preds[profile["target_quantiles"][0]] - offset
    ddt_log_q95 = ddt_log_preds[profile["target_quantiles"][2]] - offset

    # --- DDT-Hybrid ---
    ddt_hyb_p = ddt_p.copy()
    ddt_hyb_p["bin_strategy"] = "hybrid"
    ddt_hyb_p["core_fraction"] = 0.5
    
    ddt_hyb = DDTRegressor(**ddt_hyb_p)
    t0 = time.time(); ddt_hyb.fit(X_train_scaled, y_train); ddt_hyb_fit_t = time.time() - t0
    t0 = time.time()
    ddt_hyb_preds = ddt_hyb.predict_quantiles(X_test_scaled, profile["target_quantiles"])
    ddt_hyb_inf_t = time.time() - t0
    ddt_hyb_q05 = ddt_hyb_preds[profile["target_quantiles"][0]]
    ddt_hyb_q95 = ddt_hyb_preds[profile["target_quantiles"][2]]

    # --- DDT-Cal (DDT-Hybrid + marginal conformal calibration) ---
    # Calibration parameters come from profile["ddt_cal_params"].
    # DDTRegressor.fit() internally holds out `calibration_fraction` of the
    # training data; predict_quantiles() then applies per-quantile residual shifts.
    cal_overrides = profile.get("ddt_cal_params", {})
    ddt_cal_p = ddt_p.copy()
    ddt_cal_p["bin_strategy"]         = "hybrid"
    ddt_cal_p["core_fraction"]        = cal_overrides.get("core_fraction", 0.5)
    ddt_cal_p["calibration_fraction"] = cal_overrides.get("calibration_fraction", 0.15)
    ddt_cal_p["calibration_mode"]     = cal_overrides.get("calibration_mode", "marginal")

    ddt_cal = DDTRegressor(**ddt_cal_p)
    t0 = time.time(); ddt_cal.fit(X_train_scaled, y_train); ddt_cal_fit_t = time.time() - t0
    t0 = time.time()
    ddt_cal_preds = ddt_cal.predict_quantiles(X_test_scaled, profile["target_quantiles"])
    ddt_cal_inf_t = time.time() - t0
    ddt_cal_q05 = ddt_cal_preds[profile["target_quantiles"][0]]
    ddt_cal_q95 = ddt_cal_preds[profile["target_quantiles"][2]]

    # --- LightGBM ---
    lgbms = []
    t0 = time.time()
    for q in profile["target_quantiles"]:
        m = LGBMRegressor(objective="quantile", alpha=q, random_state=42, verbose=-1, **comp_p)
        m.fit(X_train_scaled, y_train); lgbms.append(m)
    lgbm_fit_t = time.time() - t0
    t0 = time.time()
    lgbm_q05 = lgbms[0].predict(X_test_scaled)
    lgbm_q95 = lgbms[2].predict(X_test_scaled)
    lgbm_inf_t = time.time() - t0

    # --- XGBoost ---
    xgbs = []
    t0 = time.time()
    for q in profile["target_quantiles"]:
        xp = comp_p.copy()
        if "num_leaves" in xp:
            xp["max_leaves"] = xp.pop("num_leaves")
            xp["grow_policy"] = "lossguide"
        m = XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, random_state=42, **xp)
        m.fit(X_train_scaled, y_train); xgbs.append(m)
    xgb_fit_t = time.time() - t0
    t0 = time.time()
    xgb_q05 = xgbs[0].predict(X_test_scaled)
    xgb_q95 = xgbs[2].predict(X_test_scaled)
    xgb_inf_t = time.time() - t0

    # --- CatBoost ---
    qs_str = ",".join(str(q) for q in profile["target_quantiles"])
    cp = comp_p.copy()
    if "n_estimators" in cp: cp["iterations"] = cp.pop("n_estimators")
    if "num_leaves"   in cp: cp.pop("num_leaves")
    cat = CatBoostRegressor(loss_function=f"MultiQuantile:alpha={qs_str}", random_seed=42, verbose=False, **cp)
    t0 = time.time(); cat.fit(X_train_scaled, y_train); cat_fit_t = time.time() - t0
    t0 = time.time(); cp_out = cat.predict(X_test_scaled); cat_inf_t = time.time() - t0
    cat_q05 = cp_out[:, 0]; cat_q95 = cp_out[:, 2]

    # --- Inverse transform ---
    if y_scaler is not None:
        def inv(a): return y_scaler.inverse_transform(a.reshape(-1, 1)).flatten()
        ddt_q05,  ddt_q95  = inv(ddt_q05),  inv(ddt_q95)
        ddt_log_q05, ddt_log_q95 = inv(ddt_log_q05), inv(ddt_log_q95)
        ddt_hyb_q05, ddt_hyb_q95 = inv(ddt_hyb_q05), inv(ddt_hyb_q95)
        ddt_cal_q05, ddt_cal_q95 = inv(ddt_cal_q05), inv(ddt_cal_q95)
        lgbm_q05, lgbm_q95 = inv(lgbm_q05), inv(lgbm_q95)
        xgb_q05,  xgb_q95  = inv(xgb_q05),  inv(xgb_q95)
        cat_q05,  cat_q95  = inv(cat_q05),  inv(cat_q95)

    # --- Metrics ---
    target_cov  = profile["target_quantiles"][2] - profile["target_quantiles"][0]
    tail_pct    = int(profile["target_quantiles"][2] * 100)
    # Fixed empirical threshold — NOT the per-sample model predicted P95
    cvar_thresh = EMPIRICAL_P95_TEST

    def fmt_row(q05, q95, inf_t, obj, fit_t):
        return [
            f"{calc_cce(y_test, q05, q95, target_cov)*100:.2f}%",
            f"{calc_apiw(q05, q95):,.4f}",
            f"{calc_cvar(y_test, q95):,.4f}",
            f"{fit_t:.2f}",
            f"{(inf_t / n_test) * 1e6:.2f}",
            f"{len(pickle.dumps(obj)) / 1024:.2f}",
            f"{calc_hei(inf_t, n_test, obj):.2f}",
        ]

    ci_lbl = f"{int(target_cov*100)}% CI"
    rows = {
        "Metric":   [
            f"CCE ({ci_lbl})", f"APIW ({ci_lbl})",
            f"CVaR (tail > predicted P{tail_pct})",
            "Train Time (s)", "Latency (us/sample)", "Model Size (KB)", "HEI",
        ],
        "DDT":      fmt_row(ddt_q05,  ddt_q95,  ddt_inf_t,  ddt,   ddt_fit_t),
        "DDT-Log":  fmt_row(ddt_log_q05, ddt_log_q95, ddt_log_inf_t, ddt_log, ddt_log_fit_t),
        "DDT-Hybrid": fmt_row(ddt_hyb_q05, ddt_hyb_q95, ddt_hyb_inf_t, ddt_hyb, ddt_hyb_fit_t),
        "DDT-Cal":    fmt_row(ddt_cal_q05, ddt_cal_q95, ddt_cal_inf_t, ddt_cal, ddt_cal_fit_t),
        "LightGBM": fmt_row(lgbm_q05, lgbm_q95, lgbm_inf_t, lgbms, lgbm_fit_t),
        "XGBoost":  fmt_row(xgb_q05,  xgb_q95,  xgb_inf_t,  xgbs,  xgb_fit_t),
        "CatBoost": fmt_row(cat_q05,  cat_q95,  cat_inf_t,  cat,   cat_fit_t),
    }
    df = pd.DataFrame(rows).set_index("Metric")
    display(HTML(f"<h4>{profile['name']} ({preproc_name})</h4>"))
    display(df)
    return f"=== {profile['name']} | {preproc_name} ===\n" + df.to_string() + "\n\n"


In [5]:
all_results_text = ""

# "winsorized" clips extreme target values to [P1, P99] then standard-scales.
# California Housing has a hard ceiling at $500k that creates extreme values;
# winsorization prevents it from dominating bin boundaries.
# Feature scaling "none" is correct: all models here are tree-based.
X_tr_s, X_te_s, y_tr_s, y_scaler = preprocess_pipeline(
    X_train_raw, X_test_raw, y_train,
    target_strategy="windsorize",
    feature_scaling="none",
)

for prof_key, prof in BENCHMARK_PROFILES.items():
    res_text = run_benchmark(prof_key, prof, X_tr_s, X_te_s, y_tr_s, y_test, "windsorize", y_scaler=y_scaler)
    all_results_text += res_text



Profile: 1-vs-1 Representational Capacity (The Theoretical Duel) | Preprocessing: windsorize


,DDT,DDT-Log,DDT-Hybrid,DDT-Cal,LightGBM,XGBoost,CatBoost
Metric,,,,,,,
CCE (89% CI),3.71%,3.03%,3.18%,0.25%,0.36%,0.79%,0.21%
APIW (89% CI),2.1218,2.1058,2.1441,2.2074,4.1206,4.1957,4.2253
CVaR (tail > predicted P95),4.1081,4.2566,4.1865,3.4437,4.9976,4.9994,4.9995
Train Time (s),0.02,0.02,0.02,0.02,0.18,0.10,0.19
Latency (us/sample),0.26,0.37,0.23,1.85,0.83,0.98,0.27
Model Size (KB),20.65,21.71,21.71,50.79,19.06,36.11,8.06
HEI,5.29,8.01,4.99,94.07,15.91,35.30,2.21



Profile: Edge AI & Micro-Kernel (The Silicon Test) | Preprocessing: windsorize


,DDT,DDT-Log,DDT-Hybrid,DDT-Cal,LightGBM,XGBoost,CatBoost
Metric,,,,,,,
CCE (89% CI),3.71%,3.03%,3.18%,0.12%,0.36%,0.79%,0.21%
APIW (89% CI),2.1218,2.1058,2.1441,2.2383,4.1206,4.1957,4.2253
CVaR (tail > predicted P95),4.1081,4.2566,4.1865,3.5057,4.9976,4.9994,4.9995
Train Time (s),0.02,0.02,0.02,0.02,0.03,0.06,0.04
Latency (us/sample),0.73,0.31,0.29,1.36,0.67,0.70,0.24
Model Size (KB),20.65,21.71,21.71,50.79,19.06,36.11,8.06
HEI,15.00,6.75,6.23,69.27,12.72,25.33,1.92



Profile: Production Battle (Single-Tree DDT vs. 100-Tree Boosting) | Preprocessing: windsorize


,DDT,DDT-Log,DDT-Hybrid,DDT-Cal,LightGBM,XGBoost,CatBoost
Metric,,,,,,,
CCE (89% CI),4.00%,3.23%,3.59%,0.15%,5.94%,8.75%,1.62%
APIW (89% CI),1.9978,2.0014,2.0210,2.1187,1.3615,1.1394,1.8262
CVaR (tail > predicted P95),4.0439,4.1319,4.1488,3.4777,3.7775,3.6470,4.1225
Train Time (s),0.03,0.02,0.03,0.02,0.29,1.52,1.40
Latency (us/sample),0.30,0.31,0.30,1.38,3.87,2.17,0.63
Model Size (KB),37.42,38.48,39.53,63.88,851.68,3601.13,821.49
HEI,11.23,11.75,11.99,87.95,3298.33,7826.82,516.12



Profile: Ceiling Tail Stress Test (DDT Core Strength) | Preprocessing: windsorize


,DDT,DDT-Log,DDT-Hybrid,DDT-Cal,LightGBM,XGBoost,CatBoost
Metric,,,,,,,
CCE (89% CI),4.85%,3.93%,4.53%,0.13%,2.38%,0.16%,2.47%
APIW (89% CI),1.8989,1.8246,1.9133,2.0257,1.6883,1.7583,2.4722
CVaR (tail > predicted P95),3.9908,4.0116,3.9613,3.3260,4.1831,4.2955,4.7272
Train Time (s),0.03,0.03,0.03,0.02,0.17,0.49,0.43
Latency (us/sample),0.32,0.36,0.27,1.52,2.02,1.56,0.38
Model Size (KB),67.83,73.70,69.79,96.26,429.79,1202.70,213.88
HEI,21.43,26.27,18.87,146.20,867.97,1875.31,80.26


In [6]:
print("--- FINAL COMPILED RESULTS ---")
print(all_results_text)


--- FINAL COMPILED RESULTS ---
=== 1-vs-1 Representational Capacity (The Theoretical Duel) | windsorize ===
                                DDT DDT-Log DDT-Hybrid DDT-Cal LightGBM XGBoost CatBoost
Metric                                                                                  
CCE (89% CI)                  3.71%   3.03%      3.18%   0.25%    0.36%   0.79%    0.21%
APIW (89% CI)                2.1218  2.1058     2.1441  2.2074   4.1206  4.1957   4.2253
CVaR (tail > predicted P95)  4.1081  4.2566     4.1865  3.4437   4.9976  4.9994   4.9995
Train Time (s)                 0.02    0.02       0.02    0.02     0.18    0.10     0.19
Latency (us/sample)            0.26    0.37       0.23    1.85     0.83    0.98     0.27
Model Size (KB)               20.65   21.71      21.71   50.79    19.06   36.11     8.06
HEI                            5.29    8.01       4.99   94.07    15.91   35.30     2.21

=== Edge AI & Micro-Kernel (The Silicon Test) | windsorize ===
                           